In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
base_url = "https://www.chavesnamao.com.br/apartamentos-a-venda/pb-joao-pessoa/?pg="
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7'
}

In [ ]:
response = requests.get(base_url, headers=headers)
print(response.status_code)

In [ ]:
def get_links_from_sitemap():
    # Nota: Você precisará confirmar a URL exata do sitemap de imóveis deles.
    # Geralmente fica listado dentro de https://www.chavesnamao.com.br/robots.txt
    sitemap_index_url = "https://www.chavesnamao.com.br/sitemap.xml" 
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    links_joao_pessoa = []
    
    print("Acessando o índice do Sitemap...")
    response = requests.get(sitemap_index_url, headers=headers)
    
    if response.status_code == 200:
        # Usamos 'xml' como parser em vez de 'html.parser'
        soup = BeautifulSoup(response.text, 'xml')
        
        # 1. Encontra todos os sub-sitemaps (sites grandes dividem o XML em vários arquivos)
        sitemaps = soup.find_all('loc')
        
        for sitemap in sitemaps:
            url_sub_sitemap = sitemap.text
            
            # Filtro opcional para não baixar sitemaps de blog ou outras cidades, se a URL permitir
            if "imoveis" in url_sub_sitemap:
                print(f"Lendo sub-sitemap: {url_sub_sitemap}")
                
                resp_sub = requests.get(url_sub_sitemap, headers=headers)
                soup_sub = BeautifulSoup(resp_sub.text, 'xml')
                
                # 2. Encontra todos os links de anúncios dentro deste sub-sitemap
                urls_imoveis = soup_sub.find_all('loc')
                
                for url_imovel in urls_imoveis:
                    link = url_imovel.text
                    # 3. Filtra apenas os links que são apartamentos em João Pessoa
                    if "/apartamentos-a-venda/pb-joao-pessoa/" in link:
                        links_joao_pessoa.append(link)
                        
    else:
        print(f"Erro ao acessar Sitemap. Status: {response.status_code}")
        
    # Remove duplicatas transformando em set e depois em list novamente
    links_unicos = list(set(links_joao_pessoa))
    
    print(f"\nTotal de links garantidos encontrados: {len(links_unicos)}")
    return links_unicos

In [ ]:
cards = get_links_from_sitemap(base_url)

In [ ]:
for i in range(len(cards)):
    print(len(cards[i]))

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
from datetime import datetime

def get_cards_list(base_url):
    cards_list = []
    i = 1
    
    # Adicionado um header básico para evitar bloqueios simples do servidor
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    while True:
        url = base_url + str(i)
        response = requests.get(url, headers=headers)

        soup = BeautifulSoup(response.text, 'html.parser')
        page_cards = soup.select('span[class*="cardContent"]')  
        
        if not page_cards:
            break
            
        cards_list.append(page_cards)
        print(f"Página {i} processada")
        
        # Limite de segurança temporário (ajuste conforme necessário)
        if i == 12:
            break
            
        i += 1
 
    return cards_list

def get_link_from_cards(cards_list):
    dominio_site = "https://www.chavesnamao.com.br"
    links_anuncios = []
    for cl in cards_list:
        for card in cl:
            card_pai_link = card.find_parent('a')
            link_relativo = card_pai_link.get('href', '') if card_pai_link else ''
            link_completo = dominio_site + link_relativo if link_relativo else ""
            if link_completo:
                links_anuncios.append(link_completo)

    return links_anuncios

def get_data_from_html(html_dos_anuncios):
    dados_finais = []

    for html_individual in html_dos_anuncios:
        soup = BeautifulSoup(html_individual, 'html.parser')
        anuncio_info = {}
        
        imagem = extrair_imagem(html_individual)
        anuncio_info['imagem'] = imagem

        titulo_tag = soup.find('h1')
        anuncio_info['Título'] = titulo_tag.get_text(strip=True) if titulo_tag else None

        preco_tag = soup.select_one("b span[class*='clamp']")
        if not preco_tag:
            preco_tag = soup.select_one("span[class*='clamp']")
        anuncio_info['Preço'] = preco_tag.get_text(strip=True) if preco_tag else None

        endereco_tag = soup.find('address')
        anuncio_info['Endereço'] = endereco_tag.get_text(strip=True) if endereco_tag else None
        
        descricao_tag = soup.find('p', attrs={'aria-label': 'descrição'})
        anuncio_info['Descrição'] = descricao_tag.get_text(strip=True) if descricao_tag else None

        # Extração DINÂMICA das listas de características
        container_principal = soup.select_one("div[class*='optionalItemsContainer']")
        if container_principal:
            secoes = container_principal.find_all('span', recursive=False)
            for secao in secoes:
                titulo_secao_tag = secao.find('b')
                if not titulo_secao_tag:
                    continue
                titulo_secao = titulo_secao_tag.get_text(strip=True)
                itens_lista = secao.select('ul li')
                lista_de_itens = [item.get_text(strip=True) for item in itens_lista]
                if lista_de_itens:
                    anuncio_info[titulo_secao] = lista_de_itens

        # Extração DINÂMICA dos atributos numéricos/principais
        lista_principal_tag = soup.find('ul', class_=lambda c: c and 'listContent' in c)
        if lista_principal_tag:
            itens_principais = lista_principal_tag.find_all('li', recursive=False)
            for item in itens_principais:
                label_tag = item.find('small')
                value_tag = item.find('b')
                
                if label_tag and value_tag:
                    label = label_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    anuncio_info[label] = value

        dados_finais.append(anuncio_info)
    
    return dados_finais

def extrair_imagem(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    picture_tag = soup.find('picture', id='ssrImage')
    if picture_tag:
        img_tag = picture_tag.find('img')
        if img_tag:
            return img_tag.get('src', '')
    
    picture_tag = soup.find('picture')
    if picture_tag:
        img_tag = picture_tag.find('img')
        if img_tag:
            return img_tag.get('src', '')

    return "Imagem não encontrada"

def extrair_numero_preco(texto):
    if not texto:
        return -1
    numero = re.sub(r'[^\d.,]', '', texto)
    numero = numero.replace('.', '').replace(',', '.')
    return float(numero) if numero else -1

def extrair_caracteristicas_imovel(caracteristicas):
    area = -1
    quartos = -1
    banheiros = -1
    garagens = -1

    for caracteristica in caracteristicas:
        if not caracteristica:
            continue
            
        texto_lower = caracteristica.lower()
        
        if 'área útil' in texto_lower or 'área total' in texto_lower:
            area = extrair_numero_caracteristica(caracteristica)
        elif 'quarto' in texto_lower:
            quartos = extrair_numero_caracteristica(caracteristica)
        elif 'banheiro' in texto_lower:
            banheiros = extrair_numero_caracteristica(caracteristica)
        elif 'garagens' in texto_lower:
            garagens = extrair_numero_caracteristica(caracteristica)
    
    return area, quartos, banheiros, garagens

def extrair_numero_caracteristica(texto):
    if not texto:
        return -1
    match = re.match(r'^(\d+)', texto.strip())
    return int(match.group(1)) if match else -1

def extrair_numero_suites(valor_suites):
    if not valor_suites or valor_suites == 'None':
        return 0
    if isinstance(valor_suites, (int, float)):
        return int(valor_suites)
    if isinstance(valor_suites, str):
        match = re.search(r'(\d+)', valor_suites)
        if match:
            return int(match.group(1))
    return 0

def get_caracteristica_especifica(key, caracteristica):
    if not key:
        return False
    return caracteristica.lower() in str(key).lower()

def get_data_from_cards(cards_list, tipo_imovel):
    link_anuncios = get_link_from_cards(cards_list)
        
    if not link_anuncios:
        print(f"Nenhum link encontrado para {tipo_imovel}!")
        return pd.DataFrame()
    
    data_atual = datetime.now()
    ano_atual = data_atual.year
    mes_atual = data_atual.month
    
    registros_extraidos = []
    
    link_to_card = {}
    dominio_site = "https://www.chavesnamao.com.br"
    for cl in cards_list:
        for card in cl:
            card_pai_link = card.find_parent('a')
            if card_pai_link:
                link_relativo = card_pai_link.get('href', '')
                link_completo = dominio_site + link_relativo if link_relativo else ""
                if link_completo in link_anuncios:
                    link_to_card[link_completo] = card
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    for i, link in enumerate(link_anuncios):
        print(f"\n🔄 Processando link {i+1}/{len(link_anuncios)}: {link}")
        
        try:
            card = link_to_card.get(link)
            if not card:
                print(f"  ⚠️ Card não encontrado para o link")
                continue
            
            target_rua = card.select_one('address p:nth-of-type(1)')
            target_bairro = card.select_one('address p:nth-of-type(2)')
            
            rua = "Não informado"
            numero = -1
            bairro = None
            
            if target_rua and target_bairro:
                endereco_completo_card = target_rua['title']
                
                if ',' in endereco_completo_card and ('Avenida' in endereco_completo_card or 'Rua' in endereco_completo_card):
                    rua = endereco_completo_card.split(',')[0].strip()
                    numero_str = endereco_completo_card.split(',')[1].strip()
                    if numero_str.isdigit():
                        numero = int(numero_str)
                else:
                    rua = endereco_completo_card.strip()
                
                bairro = target_bairro['title'].split(',')[0] if target_bairro else None
                
                if rua == "Endereço indisponível":
                    rua = "Não informado"
            
            caracteristicas_tags = card.select('span[aria-label="list"] p')
            caracteristicas = [p.get('title') for p in caracteristicas_tags if p.get('title')]
            
            area, quarto, banheiro, garagem = extrair_caracteristicas_imovel(caracteristicas)
            
            titulo_tag = card.select_one('h2[class*="contentTitle"]')
            titulo = titulo_tag.text.strip() if titulo_tag else 'Título não encontrado'
            
            preco_tag = card.select_one('p[aria-label="Preço"] b')
            preco = preco_tag.text.strip() if preco_tag else None
            condominio_tag = card.select_one('span[aria-label="Condominio"] span')
            condominio = condominio_tag.text.strip() if condominio_tag else None
            iptu_tag = card.select_one('span[aria-label="IPTU"] span')
            iptu = iptu_tag.text.strip() if iptu_tag else None
            
            preco_val = extrair_numero_preco(preco)
            preco_cond = extrair_numero_preco(condominio)
            preco_iptu_val = extrair_numero_preco(iptu)
            
            print(f"  🌐 Buscando detalhes do anúncio...")
            response = requests.get(link, headers=headers)
            if response.status_code != 200:
                print(f"  ❌ Erro ao buscar HTML: status {response.status_code}")
                continue
            
            html_anuncio = response.text
            dados_html = get_data_from_html([html_anuncio])[0]
            
            imagen_url = dados_html.get('imagem', 'Imagem não encontrada')
            
            suites_val = extrair_numero_suites(dados_html.get('Suites'))
            elevador_val = get_caracteristica_especifica(dados_html.get('Área comum'), 'elevador')
            piscina_val = get_caracteristica_especifica(dados_html.get('Área comum'), 'piscina')
            academia_val = get_caracteristica_especifica(dados_html.get('Área comum'), 'academia')
            portaria_24h_val = get_caracteristica_especifica(dados_html.get('Área comum'), 'portaria') or get_caracteristica_especifica(dados_html.get('Área comum'), '24h')
            varanda_val = get_caracteristica_especifica(dados_html.get('Área privativa'), 'varanda')
            quadra_val = get_caracteristica_especifica(dados_html.get('Área comum'), 'quadra')
            
            endereco_completo = dados_html.get('Endereço', '')
            if endereco_completo:
                endereco_completo = re.sub(r'^Endereço\s+indisponível', '', endereco_completo, flags=re.IGNORECASE).strip()
            
            # Estrutura final do dicionário
            dados_registro = {
                'preco': preco_val,
                'preco_condominio': preco_cond,
                'preco_iptu': preco_iptu_val,
                'titulo': titulo,
                'tamanho': area if area else -1,
                'quartos': quarto if quarto else -1,
                'banheiros': banheiro if banheiro else -1,
                'suites': suites_val,
                'garagens': garagem if garagem else -1,
                'elevador': elevador_val,
                'piscina': piscina_val,
                'academia': academia_val,
                'portaria_24h': portaria_24h_val,
                'varanda': varanda_val,
                'quadra': quadra_val,
                'endereco_completo': endereco_completo,
                'bairro': bairro,
                'rua': rua,
                'numero': numero,
                'url_imagem': imagen_url,
                'tipo_imovel': tipo_imovel,
                'ano_scraping': ano_atual,
                'mes_scraping': mes_atual,
                'url_anuncio': link,
                'detalhes_completos_json': json.dumps(dados_html, ensure_ascii=False)
            }
            
            registros_extraidos.append(dados_registro)
            print(f"  ✅ Registro processado com sucesso!")
            
        except Exception as e:
            print(f"  ❌ [ERRO] Falha ao processar o link. Motivo: {e}")
            continue
            
    # Converte tudo para DataFrame no final
    df = pd.DataFrame(registros_extraidos)
    if not df.empty:
        df = df.fillna(-1)
    return df

def run_chaves_scrapping():
    print("Iniciando scraper puro CHAVES NA MÃO...")
    url_casa = "https://www.chavesnamao.com.br/casas-a-venda/pb-joao-pessoa/?&pg="
    url_apto = "https://www.chavesnamao.com.br/apartamentos-a-venda/pb-joao-pessoa/?&pg="

    base_urls = [
        (url_casa, 'casa'),
        (url_apto, 'apartamento')
    ]

    df_total = pd.DataFrame()

    for url, tipo in base_urls:
        print(f"\n--- Iniciando extração de: {tipo.upper()} ---")
        cards_list = get_cards_list(url)
        df_tipo = get_data_from_cards(cards_list, tipo)
        
        if not df_tipo.empty:
            df_total = pd.concat([df_total, df_tipo], ignore_index=True)

    if not df_total.empty:
        # Salva o resultado em um arquivo CSV na pasta atual
        nome_arquivo = 'imoveis_chavesnamao.csv'
        df_total.to_csv(nome_arquivo, index=False, encoding='utf-8')
        print(f"\n🎉 PROCESSAMENTO CONCLUÍDO! {len(df_total)} anúncios salvos com sucesso no arquivo '{nome_arquivo}'.")
    else:
        print("\n⚠️ Nenhum dado foi extraído.")

# Para rodar o script diretamente:
if __name__ == "__main__":
    run_chaves_scrapping()